### 문제: camera frame 기준 큐브 중심 좌표를 space frame 기준으로 변환하기

→ ${}^{b}p$, ${}^{s}R_b$, ${}^{s}t_b$가 주어졌을 때 ${}^{s}p$를 구한다.

$$
{}^{s}p = {}^{s}R_b\,{}^{b}p + {}^{s}t_b
$$

**기호 정리**

| 수식 / 기호 | Python 변수 | 의미                                                                                             | 역할                      |
| ----------- | ----------- | ------------------------------------------------------------------------------------------------ | ------------------------- |
| $\{s\}$     | —           | space frame. 이 실습에서는 world frame                                                           | 고정된 기준 좌표계        |
| $\{b\}$     | —           | body frame. 이 실습에서는 camera frame                                                           | 카메라에 붙어 있는 좌표계 |
| ${}^{s}p$   | `s_p`       | 큐브 중심 위치를 space frame으로 표현한 좌표                                                     | 최종적으로 구할 값        |
| ${}^{b}p$   | `b_p`       | 같은 큐브 중심 위치를 camera frame으로 표현한 좌표                                               | 주어진 입력값             |
| ${}^{s}t_b$ | `s_t_b`     | 카메라 원점 위치를 space frame으로 표현한 벡터                                                   | 주어진 카메라 위치        |
| ${}^{s}R_b$ | `s_R_b`     | 카메라 축을 space frame으로 표현한 회전행렬. camera frame의 벡터 성분을 space frame으로 변환한다 | 주어진 카메라 자세        |

1. 입력값 생성
   - MuJoCo의 실제 `s_p`와 카메라 자세로 측정값 `b_p`를 만듦
   - space frame → camera frame 변환
2. 회전행렬 변환
   - 생성된 `b_p`를 입력으로 받아, 직접 작성한 회전행렬과 카메라 위치로 `s_p`를 계산
   - camera frame → space frame 변환
3. 검증
   - 계산한 `s_p`를 MuJoCo의 실제 큐브 위치와 비교

---


In [1]:
from pathlib import Path

import mujoco
import numpy as np

In [2]:
# Find the project root from the notebook working directory.
working_dir = Path.cwd().resolve()
project_root = next(
    (
        directory
        for directory in (working_dir, *working_dir.parents)
        if (directory / "pyproject.toml").is_file()
        and (directory / "models" / "camera_frame.xml").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not find models/camera_frame.xml. "
        "Start Jupyter from the project root or one of its subdirectories."
    )

model = mujoco.MjModel.from_xml_path(
    str(project_root / "models" / "camera_frame.xml")
)
data = mujoco.MjData(model)  # Simulation state and data
mujoco.mj_forward(model, data)  # Compute quantities for the current state

- 환경 불러오기


---


### 카메라의 6D pose가 주어졌다고 가정했음 (${}^{s}t_b$, ${}^{s}R_b$)


In [17]:
# Camera origin expressed in the space frame;
# the translation vector from the space origin to the camera origin.
s_t_b = data.camera("lab_camera").xpos.copy()
print("s_t_b:", s_t_b)

# Rotation matrix from the camera frame to the space frame.
s_R_b = data.camera("lab_camera").xmat.reshape(3, 3).copy()
print("s_R_b:\n", s_R_b)

s_t_b: [ 0.1  -0.55  0.64]
s_R_b:
 [[ 1.          0.          0.        ]
 [ 0.          0.70710678 -0.70710678]
 [ 0.          0.70710678  0.70710678]]


- `xpos`
  - world frame 기준의 3D 위치
  - `[x, y, z]`

- `xmat`
  - world frame 기준 회전행렬
  - 9개의 값으로 저장되므로 `reshape(3, 3)`으로 행렬 형태로 바꿈


---

### camera에서 본 cube의 위치 ($b_p$가 주어졌다고 가정했음)

(MuJoCo에서 실제 위치를 알고 있으므로 **카메라 측정값** `b_p`를 만듦)

$$
{}^{b}p = \left({}^{s}R_b\right)^{T}\left({}^{s}p - {}^{s}t_b\right)
$$

먼저 두 위치를 space frame에서 빼고, 그 상대 벡터를 camera frame 기준으로 표현한다.


In [20]:
# Cube body origin expressed in the space frame.
s_p = data.body("cube").xpos.copy()
b_p = s_R_b.T @ (s_p - s_t_b)
print("b_p:", b_p)

b_p: [ 0.00000000e+00  1.11022302e-16 -8.48528137e-01]


- ${}^{s}p - {}^{s}t_b$: "카메라에서 큐브까지 얼마나 떨어져 있는가"
- $\left({}^{s}R_b\right)^T$:
  - 상대 벡터를 camera frame 기준 성분으로 바꿈
  - 회전행렬의 Transpose는 역행렬이므로 변환 방향이 반대가 됨


---

### ${}^{s}p$ 구하기

$$
{}^{s}p = {}^{s}R_b\,{}^{b}p + {}^{s}t_b
$$


In [15]:
s_p = s_R_b @ b_p + s_t_b
print("s_p:", s_p)

s_p: [0.1  0.05 0.04]


In [16]:
s_p = data.body("cube").xpos.copy()
print("s_p:", s_p)

s_p: [0.1  0.05 0.04]
